In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve repo root whether launched from repo root or Notebooks/
repo_root = Path.cwd().resolve()
if not (repo_root / "backtesting" / "test_001_nvda").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import common

# ── Strategy configuration ───────────────────────────────────────────────
STRATEGY_DIR = repo_root / "backtesting" / "test_001_nvda"
config = common.load_strategy_config(STRATEGY_DIR / "strategy_config.json")

SYMBOL = config["stock_symbol"]
MODEL_NAME = config["model_name"]
START = config["backtest_start"]
END = config["backtest_end"]
PROVIDER = config["data_src"]

# Indicators used across all three models
INDICATOR_SPEC = {
    "rsi": {"period": 14},
    "macd": {"fast": 12, "slow": 26, "signal": 9},
    "cci": {"period": 20},
}
FEATURE_COLUMNS = ["rsi", "macd_hist", "cci"]

print(f"Strategy: {MODEL_NAME}")
print(f"Symbol:   {SYMBOL}")
print(f"Window:   {START} -> {END}")
print(f"Features: {FEATURE_COLUMNS}")

In [ ]:
# ── Fetch OHLCV data and compute indicators ─────────────────────────────
df = common.fetch_ohlcv(SYMBOL, start=START, end=END, provider=PROVIDER)
df = common.compute_indicators(df, INDICATOR_SPEC).dropna()

print(f"Bars: {len(df)}  ({df.index[0].date()} -> {df.index[-1].date()})")
df[["close", *FEATURE_COLUMNS]].tail()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# MODEL 1: Manual — indicator threshold voting (no ML)
# ══════════════════════════════════════════════════════════════════════════
# Strategy-specific rules: tune thresholds for NVDA's volatility profile.
# RSI 30/70 is standard; CCI thresholds are tighter because NVDA trends hard.
MANUAL_RULES = {
    "rsi_oversold": 30,
    "rsi_overbought": 70,
    "macd_bullish": 0,
    "cci_oversold": -100,
    "cci_overbought": 50,
    "score_buy_threshold": 2,
    "score_sell_threshold": -2,
}

manual_model = common.create_model("manual", rules=MANUAL_RULES)
manual_meta = manual_model.train(df)

# Generate signals across the full history
manual_signals = df.apply(manual_model.predict, axis=1)
print("Manual model signal distribution:")
print(manual_signals.value_counts())
print(f"\nRules: {manual_model.rules}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# MODEL 2: Classification — BagLearner(RTLearner) bagged decision trees
# ══════════════════════════════════════════════════════════════════════════
# Labels each day by its N-day forward return, then trains a bagged random
# forest to classify buy (+1) / sell (-1) / hold (0).
#
# Strategy-specific params: 10-day lookahead and 4% threshold are tuned for
# NVDA's typical swing magnitude over 2-week windows.
CLASSIFICATION_PARAMS = {
    "feature_columns": FEATURE_COLUMNS,
    "lookahead": 10,
    "threshold": 0.04,
    "leaf_size": 25,
    "bags": 15,
}

clf_model = common.create_model("classification", **CLASSIFICATION_PARAMS)
clf_meta = clf_model.train(df)

# Generate signals across the full history
clf_signals = df.apply(clf_model.predict, axis=1)
print("Classification model training metadata:")
print(clf_meta)
print(f"\nSignal distribution:")
print(clf_signals.value_counts())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# MODEL 3: Q-Learning — tabular RL with discretized indicator states
# ══════════════════════════════════════════════════════════════════════════
# Discretizes continuous indicators into bins, encodes (bins + holding) as
# a single state integer, and learns a Q-table via epsilon-greedy exploration.
#
# Strategy-specific params: 10 bins and 100 epochs balance state coverage
# vs convergence for NVDA's ~770 training bars.
QLEARNING_PARAMS = {
    "feature_columns": FEATURE_COLUMNS,
    "n_bins": 10,
    "n_epochs": 100,
    "alpha": 0.2,
    "gamma": 0.9,
    "rar": 0.98,
    "radr": 0.999,
    "dyna": 0,
}

ql_model = common.create_model("qlearning", **QLEARNING_PARAMS)
ql_meta = ql_model.train(df)

# Generate signals across the full history
ql_signals = df.apply(ql_model.predict, axis=1)
print("Q-Learning model training metadata:")
print(ql_meta)
print(f"\nSignal distribution:")
print(ql_signals.value_counts())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Compare all three models
# ══════════════════════════════════════════════════════════════════════════

def simulate_equity(signals, prices, initial_cash=100_000):
    """Simulate a long-only equity curve from buy/sell/hold signals."""
    cash = initial_cash
    shares = 0
    equity = []
    for date, signal in signals.items():
        price = prices.loc[date]
        if signal == "buy" and shares == 0:
            shares = int(cash // price)
            cash -= shares * price
        elif signal == "sell" and shares > 0:
            cash += shares * price
            shares = 0
        equity.append(cash + shares * price)
    return pd.Series(equity, index=signals.index)


models = {
    "Manual": manual_signals,
    "Classification": clf_signals,
    "Q-Learning": ql_signals,
}

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Panel 1: Normalized equity curves
for name, signals in models.items():
    eq = simulate_equity(signals, df["close"])
    axes[0].plot(eq.index, eq / eq.iloc[0], label=name)

benchmark = df["close"] / df["close"].iloc[0]
axes[0].plot(benchmark.index, benchmark, label="Buy & Hold", linestyle="--", color="gray")
axes[0].set_ylabel("Normalized Equity")
axes[0].legend()
axes[0].set_title(f"{SYMBOL} — Model Comparison")
axes[0].grid(True, alpha=0.3)

# Panel 2: Signal agreement
signal_df = pd.DataFrame(models)
agreement = signal_df.apply(lambda row: len(set(row)) == 1, axis=1)
axes[1].fill_between(signal_df.index, 0, 1, where=agreement, alpha=0.3, color="green", label="All agree")
axes[1].fill_between(signal_df.index, 0, 1, where=~agreement, alpha=0.3, color="red", label="Disagree")
axes[1].set_ylabel("Signal Agreement")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary table
summary = []
for name, signals in models.items():
    eq = simulate_equity(signals, df["close"])
    total_ret = eq.iloc[-1] / eq.iloc[0] - 1
    daily_rets = eq.pct_change().dropna()
    sharpe = daily_rets.mean() / daily_rets.std() * np.sqrt(252) if daily_rets.std() > 0 else 0
    n_trades = ((signals == "buy") | (signals == "sell")).sum()
    summary.append({"Model": name, "Return": f"{total_ret:.2%}", "Sharpe": f"{sharpe:.2f}", "Trades": n_trades})

pd.DataFrame(summary)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Export chosen model
# ══════════════════════════════════════════════════════════════════════════
# Pick the model to export for backtesting via LEAN and live trading.
# Change EXPORT_MODEL to switch which model gets exported.

EXPORT_MODEL = "classification"  # "manual", "classification", or "qlearning"

export_map = {
    "manual": manual_model,
    "classification": clf_model,
    "qlearning": ql_model,
}

model = export_map[EXPORT_MODEL]
metadata = model.save(STRATEGY_DIR, MODEL_NAME)

print(f"Exported {EXPORT_MODEL} model to {STRATEGY_DIR}/")
print(f"Artifacts: {list(metadata.get('artifacts', {}).keys())}")